# 02 · Carregando frases e gerando embeddings

Fluxo deste notebook:
1. Ler o CSV de frases de políticos.
2. Gravar políticos e frases no SQL Server.
3. Transformar cada frase em um vetor de 384 números (embedding).
4. Converter o vetor em bytes e gravar no `VARBINARY`.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore", category=UserWarning)   # aviso do pandas sobre conexão pyodbc
sys.path.append(str(pathlib.Path.cwd().parent))   # permite "from src import ..."
from src import config, banco
import pandas as pd, numpy as np
from src.vetores import vetor_para_bytes, bytes_para_vetor

conn = banco.conectar(config.SQL_DATABASE)

## 1. Os dados

In [ ]:
df = pd.read_csv("../data/frases_politicos.csv")
df = df.astype(object).where(df.notna(), None)   # NaN vira NULL no banco
print(len(df), "frases de", df.politico.nunique(), "políticos")
df[["politico", "ano", "texto"]]

## 2. Gravando políticos e frases (idempotente: pode rodar de novo)

In [ ]:
cur = conn.cursor()
for _, p in df[["politico", "pais", "cargo"]].drop_duplicates("politico").iterrows():
    cur.execute("""
        IF NOT EXISTS (SELECT 1 FROM rag.Politico WHERE Nome = ?)
            INSERT INTO rag.Politico (Nome, Pais, Cargo) VALUES (?, ?, ?)
    """, p.politico, p.politico, p.pais, p.cargo)

for _, f in df.iterrows():
    cur.execute("""
        IF NOT EXISTS (SELECT 1 FROM rag.Frase WHERE Texto = ?)
            INSERT INTO rag.Frase (PoliticoId, Texto, TextoOriginal, IdiomaOriginal, Ano, Contexto, Fonte)
            SELECT PoliticoId, ?, ?, ?, ?, ?, ? FROM rag.Politico WHERE Nome = ?
    """, f.texto, f.texto, f.texto_original, f.idioma_original,
         int(f.ano) if f.ano else None, f.contexto, f.fonte, f.politico)
conn.commit()
pd.read_sql("SELECT TOP 5 * FROM rag.vw_Frases", conn)

## 3. O modelo de embeddings

`paraphrase-multilingual-MiniLM-L12-v2`: gratuito, roda na CPU e é **multilíngue**. Na primeira execução ele baixa uns 470 MB.

Analogia: o modelo é um "GPS de significado". Ele coloca cada frase num ponto de um mapa com 384 eixos. Frases com sentido parecido ficam perto, mesmo usando palavras diferentes.

In [ ]:
from sentence_transformers import SentenceTransformer
modelo = SentenceTransformer(config.MODELO_EMBEDDING)
DIM = modelo.get_sentence_embedding_dimension()
print("Dimensões:", DIM)

In [ ]:
v = modelo.encode("A única coisa que devemos temer é o próprio medo.", normalize_embeddings=True)
print("Primeiros 8 números:", np.round(v[:8], 4))
print("Norma (tamanho do vetor):", round(float(np.linalg.norm(v)), 4))

`normalize_embeddings=True` deixa todo vetor com norma 1. Vantagem: o cosseno vira uma simples multiplicação e soma (produto escalar).

## 4. Vetor → bytes: a "mágica" que dispensa o tipo VECTOR

In [ ]:
b = vetor_para_bytes(v)
print("Bytes:", len(b), "=", DIM, "dimensões x 4 bytes")
print("Primeiros 16 bytes:", b[:16].hex(" "))
print("Volta sem perda?", np.array_equal(bytes_para_vetor(b), v.astype("<f4")))

### E se eu guardasse como JSON em NVARCHAR?

In [ ]:
import json
como_json = json.dumps([float(x) for x in v])
print(f"VARBINARY: {len(b):>6} bytes")
print(f"NVARCHAR (JSON): {len(como_json.encode('utf-16-le')):>6} bytes")
print(f"JSON ocupa {len(como_json.encode('utf-16-le'))/len(b):.1f}x mais espaço, e ainda precisa de parse.")

## 5. Gerando e gravando os embeddings de todas as frases

In [ ]:
frases = pd.read_sql("SELECT FraseId, Texto FROM rag.Frase ORDER BY FraseId", conn)
vetores = modelo.encode(frases.Texto.tolist(), normalize_embeddings=True, show_progress_bar=True)

cur.execute("DELETE FROM rag.FraseEmbedding WHERE Modelo = ?", config.MODELO_EMBEDDING)
cur.executemany(
    "INSERT INTO rag.FraseEmbedding (FraseId, Modelo, Dimensoes, Vetor) VALUES (?, ?, ?, ?)",
    [(int(fid), config.MODELO_EMBEDDING, DIM, vetor_para_bytes(vec))
     for fid, vec in zip(frases.FraseId, vetores)],
)
conn.commit()
print(len(frases), "embeddings gravados")

## 6. Conferindo pelo lado do DBA

In [ ]:
pd.read_sql("""
SELECT TOP 5 e.FraseId, e.Dimensoes, DATALENGTH(e.Vetor) AS Bytes,
       CONVERT(VARCHAR(40), SUBSTRING(e.Vetor, 1, 16), 1) AS PrimeirosBytes, e.CriadoEm
FROM rag.FraseEmbedding e
""", conn)

### A constraint funciona? Tentando gravar um vetor com tamanho errado

In [ ]:
try:
    cur.execute("INSERT INTO rag.FraseEmbedding (FraseId, Modelo, Dimensoes, Vetor) VALUES (1, 'teste', 384, ?)",
                vetor_para_bytes(v[:100]))
    conn.commit()
except Exception as e:
    conn.rollback()
    print("Bloqueado pelo CHECK:", str(e)[:160])

## 7. Bônus: explodindo o vetor em linhas (para o cálculo em T-SQL puro)

In [ ]:
cur.execute("DELETE FROM rag.FraseEmbeddingItem WHERE Modelo = ?", config.MODELO_EMBEDDING)
linhas = [(int(fid), config.MODELO_EMBEDDING, d, float(val))
          for fid, vec in zip(frases.FraseId, vetores) for d, val in enumerate(vec)]
cur.fast_executemany = True
cur.executemany("INSERT INTO rag.FraseEmbeddingItem (FraseId, Modelo, Dim, Valor) VALUES (?, ?, ?, ?)", linhas)
cur.fast_executemany = False
conn.commit()
print(f"{len(linhas):,} linhas ({len(frases)} frases x {DIM} dimensões)")